In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests


BASE = Path(r"H:\ASD projoect")

# Your EF preprocessing is currently using RF
EF_RECORDING = "CF"

In [ ]:
## ASD ("1170","3200"),("1161","3081"),("1121","3041"),("1101","3061"),
    # ("1141","3161"),("1151","3071"),("1191","3111"),("1031","3121"),
    # ("1021","3021"),("1011","3011"),("1071","3051"),
    
# TD # ("2360","4150"),("2121","4181"),("2201","4171"),("2250","4210"),
    # ("2281","4091"),("2340","4080"),("2151","4071"),("2211","4141"),

asd_sub_num_ef = [
    1361, 1531, 1451, 1521, 1470, 1381, 1111, 1281, 1341, 
    1091, 1061, 1311, 1481, 1711, 1671, 1720, 1741, 1811, 
    1761, 1771, 1790, 1901, 1690, 1871, 1861, 1881, 1801, 
    1851, 1821, 1780,
    1201, 1050, 1131, 1080, 1041, 1551, 1431, 1181, 1351, 
    1491, 1511, 1211, 1421, 1321, 1231, 1571, 1641, 1581,
    1621, 1681, 1651,
    1170, 1161, 1121, 1101, 1141, 1151, 1191, 1031,
    1021, 1011, 1071,
    1261, 1371, 1441, 1271, 1241, 1301, 1541, 1461, 1501,
    1391, 1701, 1411, 1561, 1611, 1631, 1591, 1731, 1601,
    1661, 1780, 1331, 1401, 1291, 1220, 1251
]

asd_sub_num_tom = [
    3431, 3301, 3390, 3441, 3361, 3381, 3400, 3261, 3520, 
    3711, 3690, 3291, 3371, 3351, 3331, 3311, 3271, 3191, 
    3321, 3101, 3171, 3480, 3551, 3471, 3591, 3561, 3531, 
    3671, 3511, 3741, 3631, 3751, 3640, 3730, 3781, 3771, 
    3701, 3721, 3711,
    3301, 3390, 3381, 3400, 3261, 3291, 3371, 3351, 3331, 
    3311, 3271, 3191, 3321, 3101, 3171, 3471, 3591, 3561, 
    3531, 3671, 3631,
    3200, 3081, 3041, 3061, 3161, 3071, 3111, 3121,
    3021, 3011, 3051,
    3571, 3231, 3421, 3091, 3211, 3181, 3141, 3241, 3361,
    3131, 3601, 3281, 3031, 3501, 3541, 3491, 3611, 3581,
    3461, 3450, 3411, 3221, 3151, 3250, 3341
]


td_sub_num_ef = [
    2491, 2010, 2101, 2021, 2070, 2111, 2580, 2261, 2410, 
    2061, 2590, 2610, 2560, 2661, 2651, 2571, 2620, 2600, 
    2691, 2631, 2671, 2641, 2681,
    2051, 2080, 2090, 2031, 2401, 2160, 2450, 2270, 2190,
    2461, 2220, 2540, 2550,
    2360, 2121, 2201, 2250, 2281, 2340, 2151, 2211,
    2180, 2500, 2231, 2510, 2480, 2441, 2371, 2401, 2470,
    2420, 2171, 2331, 2380, 2390, 2310, 2240, 2521, 2531,
    2321, 2430, 2291, 2350, 2301
]

td_sub_num_tom = [
    4260, 4430, 4281, 4120, 4390, 4011, 4380, 4581, 4441, 
    4031, 4361, 4421, 4290, 4240, 4271, 4221, 4331, 4480, 
    4560, 4320, 4231, 4541, 4340, 4500, 4451, 4400, 4410, 
    4160, 4110, 4471, 4511, 4351, 4370, 4301, 4531, 4461, 
    4520, 4130, 4491, 4250, 4601, 4610, 4640, 4631, 4550, 
    4570, 4591, 4701, 4660, 4651, 4620, 4671, 4680, 4691, 
    4710, 4721, 4731, 4751,
    4361, 4290, 4240, 4271, 4331, 4340, 4500, 4520, 4130, 
    4491, 4250, 4550, 4570,
    4150, 4181, 4171, 4210, 4091, 4080, 4071, 4141,
    4260, 4430, 4281, 4390, 4380, 4441, 4421, 4331, 4480,
    4320, 4231, 4451, 4400, 4410, 4160, 4110, 4471, 4511,
    4351, 4370, 4301, 4531, 4461
]


PAIRS = {
    "ASD": list(zip(asd_sub_num_ef, asd_sub_num_tom)),
    "TD": list(zip(td_sub_num_ef, td_sub_num_tom)),
}

print("ASD pairs:", len(PAIRS["ASD"]))
print("TD pairs:", len(PAIRS["TD"]))

ASD pairs: 36
TD pairs: 31


In [4]:
def load_ef_erp(file_path):

    d = np.load(
        file_path,
        allow_pickle=True
    ).item()

    return {
        "EF_delta_N2":
            d["N2"]["nogo_go_effect_uv"],

        "EF_delta_P3_LPP":
            d["P3_LPP"]["nogo_go_effect_uv"],

        "EF_N2_go":
            d["N2"]["go_amplitude_uv"],

        "EF_N2_nogo":
            d["N2"]["nogo_amplitude_uv"],

        "EF_P3_LPP_go":
            d["P3_LPP"]["go_amplitude_uv"],

        "EF_P3_LPP_nogo":
            d["P3_LPP"]["nogo_amplitude_uv"],
    }
    
def load_tom_erp(file_path, prefix):

    d = np.load(
        file_path,
        allow_pickle=True
    ).item()

    components = [
        "LPC",
        "early_LSW",
        "mid_LSW",
        "late_LSW"
    ]

    out = {}

    for component in components:

        for domain in ["cog", "aff"]:

            key = f"delta_{component}_{domain}"

            out[
                f"{prefix}_{key}"
            ] = d.get(
                key,
                np.nan
            )

    return out

In [5]:
def load_psd(file_path, prefix):

    df = pd.read_csv(
        file_path
    )

    out = {}

    for _, row in df.iterrows():

        condition = row["condition"]
        band = row["band"]
        roi = row["roi"]

        col = (
            f"{prefix}_"
            f"{condition}_"
            f"{band}_"
            f"{roi}_PSD_dB"
        )

        out[col] = row["psd_db"]

    return out

In [6]:
def load_one_subject(
    group,
    ef_id,
    tom_id
):

    ef_dir = (
        BASE
        / "EF"
        / group
        / f"{ef_id}_{EF_RECORDING}_CV_processed_numpy"
    )

    tom_dir = (
        BASE
        / "ToM"
        / group
        / f"{tom_id}a_CV_processed_numpy"
    )

    # ---------- files ----------
    ef_erp_file = (
        ef_dir
        / "EF_ERP_results.npy"
    )

    ef_psd_file = (
        ef_dir
        / "EF_PSD_DE_condition_features.csv"
    )

    h_erp_file = (
        tom_dir
        / "ToM_ERP_hearing_results.npy"
    )

    t_erp_file = (
        tom_dir
        / "ToM_ERP_thinking_results.npy"
    )

    h_psd_file = (
        tom_dir
        / "ToM_hearing_PSD_DE_condition_features.csv"
    )

    t_psd_file = (
        tom_dir
        / "ToM_thinking_PSD_DE_condition_features.csv"
    )

    required = [
        ef_erp_file,
        ef_psd_file,
        h_erp_file,
        t_erp_file,
        h_psd_file,
        t_psd_file,
    ]

    missing = [
        f for f in required
        if not f.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing: "
            + ", ".join(
                f.name for f in missing
            )
        )

    row = {
        "group": group,
        "EF_ID": str(ef_id),
        "ToM_ID": str(tom_id),
    }

    # ERP
    row.update(
        load_ef_erp(
            ef_erp_file
        )
    )

    row.update(
        load_tom_erp(
            h_erp_file,
            "H"
        )
    )

    row.update(
        load_tom_erp(
            t_erp_file,
            "T"
        )
    )

    # PSD
    row.update(
        load_psd(
            ef_psd_file,
            "EF"
        )
    )

    row.update(
        load_psd(
            h_psd_file,
            "H"
        )
    )

    row.update(
        load_psd(
            t_psd_file,
            "T"
        )
    )

    return row

In [7]:
def load_group(group):

    rows = []
    problems = []

    pairs = PAIRS[group]

    for i, (ef_id, tom_id) in enumerate(
        pairs,
        start=1
    ):

        print(
            f"[{i}/{len(pairs)}] "
            f"{group}: "
            f"EF {ef_id} / ToM {tom_id}"
        )

        try:

            row = load_one_subject(
                group,
                ef_id,
                tom_id
            )

            rows.append(row)

            print("  loaded")

        except Exception as e:

            print(
                "  skipped:",
                type(e).__name__,
                str(e)
            )

            problems.append({
                "group": group,
                "EF_ID": ef_id,
                "ToM_ID": tom_id,
                "error": str(e)
            })

    return (
        pd.DataFrame(rows),
        pd.DataFrame(problems)
    )

In [8]:
ASD_features, ASD_problems = load_group(
    "ASD"
)

TD_features, TD_problems = load_group(
    "TD"
)

print("\nASD loaded:", len(ASD_features))
print("TD loaded:", len(TD_features))

[1/36] ASD: EF 1170 / ToM 3200
  loaded
[2/36] ASD: EF 1161 / ToM 3081
  loaded
[3/36] ASD: EF 1121 / ToM 3041
  loaded
[4/36] ASD: EF 1101 / ToM 3061
  loaded
[5/36] ASD: EF 1141 / ToM 3161
  loaded
[6/36] ASD: EF 1151 / ToM 3071
  loaded
[7/36] ASD: EF 1191 / ToM 3111
  loaded
[8/36] ASD: EF 1031 / ToM 3121
  loaded
[9/36] ASD: EF 1021 / ToM 3021
  loaded
[10/36] ASD: EF 1011 / ToM 3011
  loaded
[11/36] ASD: EF 1071 / ToM 3051
  loaded
[12/36] ASD: EF 1261 / ToM 3571
  loaded
[13/36] ASD: EF 1371 / ToM 3231
  loaded
[14/36] ASD: EF 1441 / ToM 3421
  loaded
[15/36] ASD: EF 1271 / ToM 3091
  loaded
[16/36] ASD: EF 1241 / ToM 3211
  loaded
[17/36] ASD: EF 1301 / ToM 3181
  loaded
[18/36] ASD: EF 1541 / ToM 3141
  loaded
[19/36] ASD: EF 1461 / ToM 3241
  skipped: FileNotFoundError Missing: EF_ERP_results.npy, EF_PSD_DE_condition_features.csv, ToM_hearing_PSD_DE_condition_features.csv, ToM_thinking_PSD_DE_condition_features.csv
[20/36] ASD: EF 1501 / ToM 3361
  loaded
[21/36] ASD: EF 1391

In [9]:
ASD_features.to_csv(
    BASE / "ASD_Pearson_features.csv",
    index=False
)

TD_features.to_csv(
    BASE / "TD_Pearson_features.csv",
    index=False
)

In [10]:
ERP_PAIRINGS = [
    (
        "EF_delta_N2",
        "LPC"
    ),
    (
        "EF_delta_P3_LPP",
        "early_LSW"
    ),
    (
        "EF_delta_P3_LPP",
        "mid_LSW"
    ),
    (
        "EF_delta_P3_LPP",
        "late_LSW"
    ),
]

In [ ]:
def safe_pearson(x, y):

    x = pd.to_numeric(
        x,
        errors="coerce"
    )

    y = pd.to_numeric(
        y,
        errors="coerce"
    )

    valid = (
        x.notna()
        & y.notna()
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 4:
        return len(x), np.nan, np.nan

    if (
        x.std() == 0
        or y.std() == 0
    ):
        return len(x), np.nan, np.nan

    r, p = pearsonr(
        x,
        y
    )

    return len(x), r, p


In [12]:
def erp_pearson(
    df,
    group,
    segment
):

    # segment = H or T

    rows = []

    for domain in [
        "cog",
        "aff"
    ]:

        for ef_col, tom_component in ERP_PAIRINGS:

            tom_col = (
                f"{segment}_delta_"
                f"{tom_component}_"
                f"{domain}"
            )

            if (
                ef_col not in df.columns
                or tom_col not in df.columns
            ):
                continue

            n, r, p = safe_pearson(
                df[ef_col],
                df[tom_col]
            )

            rows.append({
                "group": group,
                "segment": segment,
                "domain": domain,
                "EF_feature": ef_col,
                "ToM_feature": tom_col,
                "n": n,
                "r": r,
                "p": p
            })

    result = pd.DataFrame(
        rows
    )

    valid_p = result["p"].notna()

    result.loc[
        valid_p,
        "p_FDR"
    ] = multipletests(
        result.loc[
            valid_p,
            "p"
        ],
        method="fdr_bh"
    )[1]

    return result.sort_values(
        "p"
    ).reset_index(drop=True)

In [13]:
ASD_H_ERP = erp_pearson(
    ASD_features,
    "ASD",
    "H"
)

ASD_T_ERP = erp_pearson(
    ASD_features,
    "ASD",
    "T"
)

TD_H_ERP = erp_pearson(
    TD_features,
    "TD",
    "H"
)

TD_T_ERP = erp_pearson(
    TD_features,
    "TD",
    "T"
)

In [17]:
SAVE_DIR = (
    BASE
    / "Pearson_results"
)

SAVE_DIR.mkdir(
    exist_ok=True
)

ASD_H_ERP.to_csv(
    SAVE_DIR / "ASD_H_ERP_Pearson.csv",
    index=False
)

ASD_T_ERP.to_csv(
    SAVE_DIR / "ASD_T_ERP_Pearson.csv",
    index=False
)

TD_H_ERP.to_csv(
    SAVE_DIR / "TD_H_ERP_Pearson.csv",
    index=False
)

TD_T_ERP.to_csv(
    SAVE_DIR / "TD_T_ERP_Pearson.csv",
    index=False
)

# PSD

In [15]:
BANDS = [
    "theta",
    "alpha",
    "low_beta",
    "beta"
]

ROIS = [
    "frontal",
    "frontocentral",
    "central",
    "centroparietal",
    "parietal",
    "posterior"
]

In [16]:
def center_condition(
    df,
    prefix,
    condition
):

    for band in BANDS:

        cols = [
            f"{prefix}_{condition}_"
            f"{band}_{roi}_PSD_dB"
            for roi in ROIS
        ]

        available = [
            c for c in cols
            if c in df.columns
        ]

        if not available:
            continue

        global_mean = (
            df[available]
            .mean(axis=1)
        )

        for roi in ROIS:

            source = (
                f"{prefix}_{condition}_"
                f"{band}_{roi}_PSD_dB"
            )

            if source not in df.columns:
                continue

            df[
                f"{prefix}_{condition}_"
                f"{band}_{roi}_centered"
            ] = (
                df[source]
                - global_mean
            )

    return df

In [18]:
def center_all_psd(df):

    df = df.copy()

    for condition in [
        "go",
        "nogo"
    ]:
        df = center_condition(
            df,
            "EF",
            condition
        )

    for prefix in [
        "H",
        "T"
    ]:

        for condition in [
            "false_cog",
            "true_cog",
            "false_aff",
            "true_aff"
        ]:

            df = center_condition(
                df,
                prefix,
                condition
            )

    return df

In [19]:
def residualize(
    target,
    control
):

    target = pd.to_numeric(
        target,
        errors="coerce"
    )

    control = pd.to_numeric(
        control,
        errors="coerce"
    )

    valid = (
        target.notna()
        & control.notna()
    )

    out = pd.Series(
        np.nan,
        index=target.index,
        dtype=float
    )

    if valid.sum() < 4:
        return out

    x = control[valid].to_numpy()
    y = target[valid].to_numpy()

    if np.std(x) == 0:
        return out

    slope, intercept = np.polyfit(
        x,
        y,
        1
    )

    out.loc[valid] = (
        y
        - (
            intercept
            + slope * x
        )
    )

    return out

In [20]:
def create_specific_psd(df):

    df = center_all_psd(
        df
    )

    # ---------- EF: Nogo controlling Go ----------
    for band in BANDS:

        for roi in ROIS:

            nogo = (
                f"EF_nogo_"
                f"{band}_{roi}_centered"
            )

            go = (
                f"EF_go_"
                f"{band}_{roi}_centered"
            )

            if (
                nogo in df.columns
                and go in df.columns
            ):

                df[
                    f"EF_specific_"
                    f"{band}_{roi}"
                ] = residualize(
                    df[nogo],
                    df[go]
                )

    # ---------- ToM: False controlling True ----------
    for segment in [
        "H",
        "T"
    ]:

        for domain in [
            "cog",
            "aff"
        ]:

            for band in BANDS:

                for roi in ROIS:

                    false = (
                        f"{segment}_false_{domain}_"
                        f"{band}_{roi}_centered"
                    )

                    true = (
                        f"{segment}_true_{domain}_"
                        f"{band}_{roi}_centered"
                    )

                    if (
                        false in df.columns
                        and true in df.columns
                    ):

                        df[
                            f"{segment}_specific_"
                            f"{domain}_"
                            f"{band}_{roi}"
                        ] = residualize(
                            df[false],
                            df[true]
                        )

    return df

In [21]:
ASD_features = create_specific_psd(
    ASD_features
)

TD_features = create_specific_psd(
    TD_features
)

In [22]:
def psd_pearson(
    df,
    group,
    segment,
    domain
):

    rows = []

    for band in BANDS:

        for roi in ROIS:

            ef_col = (
                f"EF_specific_"
                f"{band}_{roi}"
            )

            tom_col = (
                f"{segment}_specific_"
                f"{domain}_"
                f"{band}_{roi}"
            )

            if (
                ef_col not in df.columns
                or tom_col not in df.columns
            ):
                continue

            n, r, p = safe_pearson(
                df[ef_col],
                df[tom_col]
            )

            rows.append({
                "group": group,
                "segment": segment,
                "domain": domain,
                "band": band,
                "roi": roi,
                "n": n,
                "r": r,
                "p": p
            })

    result = pd.DataFrame(
        rows
    )

    valid_p = result["p"].notna()

    result.loc[
        valid_p,
        "p_FDR"
    ] = multipletests(
        result.loc[
            valid_p,
            "p"
        ],
        method="fdr_bh"
    )[1]

    return result.sort_values(
        "p"
    ).reset_index(drop=True)

In [23]:
PSD_RESULTS = {}

for group, df in [
    ("ASD", ASD_features),
    ("TD", TD_features)
]:

    for segment in [
        "H",
        "T"
    ]:

        for domain in [
            "cog",
            "aff"
        ]:

            name = (
                f"{group}_"
                f"{segment}_"
                f"{domain}"
            )

            PSD_RESULTS[name] = psd_pearson(
                df,
                group,
                segment,
                domain
            )

In [27]:
for name, result in PSD_RESULTS.items():

    result.to_csv(
        SAVE_DIR
        / f"{name}_PSD_Pearson.csv",
        index=False
    )